In [32]:
# !pip install bitarray
# !pip install mmh3


In [33]:
from bitarray import bitarray
import hashlib
from hashlib import sha3_256, sha256, blake2b
import math 
import mmh3
import string
import json
import requests

In [34]:
#Asked ChatGPT to explain bitarray library, fixed number of bits, salting, and hashes
#later functions weren't working and asked ChatGPT if I was adding words to the bloom filter right. Modified this code

words = []

with open('words.txt', 'r') as file:
    for line in file:
        word = line.strip()
        words.append(word)

word_set = set(words)
        

In [35]:
#Implement filter
# Asked ChatGPT what to edit to change bit size and hashes

class BloomFilter:
    def __init__(self, size, hash_functions): #now uses any hash function
        #items_count number of items expected to be sotred in bloom filter
        self.size = size #set size asspecified in bloom
        self.bit_array = bitarray(self.size) #bit array of given size
        self.bit_array.setall(0) #initialize all bits as 0
        self.hash_functions = hash_functions #allows provided hashing
    def add(self, item):
        for func in self.hash_functions:
            digest = func(item) % self.size #each hash produces integer from item within bounds of bit array
            self.bit_array[digest] = 1 #then sets that bit to 1
    def check(self, item):
        return all(self.bit_array[func(item) % self.size] for func in self.hash_functions) #checks if item is in filter
    

In [42]:
# For each word in the word list, apply all three hash functions
# set corresponding bits in bitarray to 1

def my_hash(s):
    return int(sha256(s.lower().encode()).hexdigest(), 16) % size
def my_hash2(s):
    return int(blake2b(s.lower().encode()).hexdigest(), 16) % size
def my_hash3(s):
    return int(sha3_256(s.lower().encode()).hexdigest(), 16) % size

three_hash_functions = [my_hash, my_hash2, my_hash3]
two_hash_functions = [my_hash, my_hash2]
one_hash_function = [my_hash]

bloomf_3 = BloomFilter(1000000,three_hash_functions)
bloomf_2 = BloomFilter(1000000, two_hash_functions)
bloomf_1 = BloomFilter(1000000, one_hash_function)

for word in words:
    bloomf_3.add(word)
    bloomf_2.add(word)
    bloomf_1.add(word)

In [43]:
#b. Create a function that checks all possible single-character substitutions for a given word using the Bloom filter. 
# Return words flagged by the filter as potential matches. 

#I need the function to take in each word, replace a single character, compare to the rest of list and return if it is a match. Repeat for all letter combinations

#Asked ChatGPT how to make a function that replces single character in given word
def single_char_changes(word): 
    replaced_words = []
    for i in range(len(word)):
        for letter in string.ascii_lowercase:
            if word[i] != letter:
                changed = word[:i] + letter + word[i+1:]
                replaced_words.append(changed)
    return replaced_words

#Tested that the function does single character substitution for word given
print(single_char_changes('cat'))
len(single_char_changes('cat'))


['aat', 'bat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'cbt', 'cct', 'cdt', 'cet', 'cft', 'cgt', 'cht', 'cit', 'cjt', 'ckt', 'clt', 'cmt', 'cnt', 'cot', 'cpt', 'cqt', 'crt', 'cst', 'ctt', 'cut', 'cvt', 'cwt', 'cxt', 'cyt', 'czt', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'can', 'cao', 'cap', 'caq', 'car', 'cas', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz']


75

In [70]:
#create full function
#Asked ChatGPT how to compare again the bloom filter and ensure function is doing what i want. 
def spell_check(word, bloom):
    candidates = single_char_changes(word)
    matches = []
    print(f"\nChecking spellings for: '{word}'")
    print(f"Total candidates to check: {len(candidates)}")
    for candidate in candidates:
        if bloom.check(candidate): #check if hash function is there 
            print(f"{candidate} is a match in bloom")
            matches.append(candidate)
        else: 
            print(f"{candidate} is not in Bloom")
    return matches and print(f"there are {len(matches)} possibilities")


In [71]:
matches = spell_check("floeer", bloomf_3)
print("Possible matches from Bloom filter:", matches)



Checking spellings for: 'floeer'
Total candidates to check: 150
aloeer is a match in bloom
bloeer is a match in bloom
cloeer is a match in bloom
dloeer is a match in bloom
eloeer is a match in bloom
gloeer is a match in bloom
hloeer is a match in bloom
iloeer is a match in bloom
jloeer is a match in bloom
kloeer is a match in bloom
lloeer is a match in bloom
mloeer is a match in bloom
nloeer is not in Bloom
oloeer is a match in bloom
ploeer is a match in bloom
qloeer is a match in bloom
rloeer is a match in bloom
sloeer is a match in bloom
tloeer is a match in bloom
uloeer is a match in bloom
vloeer is a match in bloom
wloeer is a match in bloom
xloeer is a match in bloom
yloeer is a match in bloom
zloeer is a match in bloom
faoeer is a match in bloom
fboeer is a match in bloom
fcoeer is a match in bloom
fdoeer is a match in bloom
feoeer is a match in bloom
ffoeer is a match in bloom
fgoeer is a match in bloom
fhoeer is a match in bloom
fioeer is a match in bloom
fjoeer is not in Bloo

In [44]:
# Implement a function to test how well the Bloom filter suggests corrections. 
# A suggestion list is considered "good" if it contains no more than three suggestions and includes the correct word.

#https://www.geeksforgeeks.org/python/read-json-file-using-python/

#call in typos and visualize
with open('typos.json', 'r') as file2:
    typos = json.load(file2)

In [47]:
print(typos[:5])

[['soprabi', 'soprani'], ['rosan', 'rosan'], ['jeresiologer', 'heresiologer'], ['wrinkvy', 'wrinkly'], ['seaweeds', 'seaweeds']]


In [50]:
# make list of all words with replace all characters
# for each word in that list, check if its in the bloom
# return amount of words from list that are in bloom (possible suggestions)
# return if correct word is in that list and how many words returned as possible suggestions
# if possible suggestions is more than 3, bad outcome. If less than 3, good outcome

#Used ChatGPT to integrate steps in right order and understand when to call in typos. Also to fix indentation error

# Generate all single-character substitutions
def single_char_changes(word): #with wanted word, generate all single character substitutions
    replaced_words = []
    for i in range(len(word)):
        for letter in string.ascii_lowercase:
            if word[i] != letter:
                changed = word[:i] + letter + word[i+1:]
                replaced_words.append(changed)
    return replaced_words

# Check if substitutions are in bloom
def check_bloom_all_spellings(typed_word, bloom):
    candidates = single_char_changes(typed_word) # invokes single character changes on typed word and puts in candidates
    matches = []
    for candidate in candidates: #cycles through each single character change and stores in matches if it is in the word list of that bloom filter
        if bloom.check(candidate):  #generic bloom to envoke whatever was put in argument
            matches.append(candidate)
    return matches

# check if good bloom and track misidentified and good 
def good_bloom(bloom, typo, word_set): #takes in bloom, typos word list, and word list in set
    good = 0
    total = len(typo)
    misidentified = 0
    total_checked = 0

    for typed_word, correct_word in typo:
        suggestions = check_bloom_all_spellings(typed_word, bloom)  #invokes check all spelling of typed word in bloom
        total_checked += len(suggestions)

        # check if it's a good suggestion
        if len(suggestions) <= 3 and correct_word in suggestions: #and correct_word in matches: # is matches is more than 3, deems it a bad outcome. If matches is less than 3 deems it a good outcome; need to add if it includes correct word is good, if not is bad. 
            good += 1

        # Count misidentified
        for suggestion in suggestions:
            if suggestion not in word_set:
                misidentified += 1

    return good, misidentified, total, total_checked


In [51]:

good_bloom(bloomf_1, 'floeer', word_set)

ValueError: not enough values to unpack (expected 2, got 1)

In [ ]:
# EVERYTHING UNDER IS FIRST ATTEMPT

In [ ]:
#create full function
#Asked ChatGPT how to compare again the bloom filter and ensure function is doing what i want. 
# def spell_check(word):
    def single_char_changes(word): 
        replaced_words = []
        for i in range(len(word)):
            for letter in string.ascii_lowercase:
                if word[i] != letter:
                    changed = word[:i] + letter + word[i+1:]
                    replaced_words.append(changed)
        return replaced_words
    candidates = single_char_changes(word)
    for candidate in candidates:
        my_hash(candidate) = 1  # convert string to hash via function
        if  in bloomf: #check  if hash function is there 
            print(f"{candidate} is a Match!")

In [ ]:
#test function
# spell_check('cat')

eat is a Match!
qat is a Match!
rat is a Match!
xat is a Match!
yat is a Match!
zat is a Match!
cwt is a Match!
cag is a Match!
caw is a Match!
cay is a Match!


In [ ]:
# to test how well the bloom filter suggests corrections, i want to build on the function I made
# that function will be nested in another one so that it returns the suggestions and dictates if it's good or not by giving amount of suggestions 
# it needs to know if its correct by having the second word in the pair from typos? No second word needs to be in list. Typos is given to use as testing data
# 
# spell_check('wrinkvy')

wrinkly is a Match!


In [ ]:
# make list of all words with replace all characters
# for each word in that list, check if its in the bloom
# return amount of words from list that are in bloom (possible suggestions)
# return if correct word is in that list and how many words returned as possible suggestions
# if possible suggestions is more than 3, bad outcome. If less than 3, good outcome

# def good_bloom(typed_word, correct_word, bloom):
  def single_char_changes(typed_word): 
        replaced_words_bloom = []
        for i in range(len(typed_word)):
            for letter in string.ascii_lowercase:
                if typed_word[i] != letter:
                    changed = typed_word[:i] + letter + typed_word[i+1:]
                    replaced_words_bloom.append(changed)
        return replaced_words_bloom
  candidates = single_char_changes(typed_word) # puts all single character changes in list as candidates to be a match
  matches = []
  for candidate in candidates: #cycles through each single character change and stores in matches if it is in the word list
    if candidate in words:
        matches.append(candidate)
  print(matches)
  if len(matches) <= 3 and correct_word in matches: #and correct_word in matches: # is matches is more than 3, deems it a bad outcome. If matches is less than 3 deems it a good outcome; need to add if it includes correct word is good, if not is bad. 
      return (f"This is a good outcome, it has {len(matches)} matches and includes the correct word)")
  else:
      return ("This is a bad outcome")
  
  

In [ ]:
# def good_bloom(typed_word, correct_word, bloom):
    for typed_word, correct_word in typos:
        if typed_word == correct_word:
            single_char_changes(typed_word)
            

In [ ]:
# good_bloom('wrinkvy', 'wrinkly',bloomf ) #test with wrinkly which only has 1 match, good outcome

['wrinkly']


'This is a good outcome, it has 1 matches and includes the correct word)'

In [ ]:
# good_bloom('eat', 'cat', bloomf) #test with eat/cat bad outcome

['qat', 'rat', 'xat', 'yat', 'zat', 'eft', 'ent', 'est', 'ext', 'ead', 'ean', 'ear']


'This is a bad outcome'

In [ ]:
# Experiment with different Bloom filter sizes and combinations of 1, 2, or 3 hash functions. 
#create 1 bloom for 1 hash size; 1 bloom for 2 hash size; 1 bloom for 3 hash size

# Asked ChatGPT what to edit to change bit size and hashes

# class BloomFilter_new:
    def __init__(self, size, hash_functions): #now uses any hash function
        #items_count number of items expected to be sotred in bloom filter
        self.size = size #set size asspecified in bloom
        self.bit_array = bitarray(self.size) #bit array of given size
        self.bit_array.setall(0) #initialize all bits as 0
        self.hash_functions = hash_functions #allows provided hashing
    def add(self, item):
        for func in self.hash_functions:
            digest = func(item) % self.size #each hash produces integer from item within bounds of bit array
            self.bit_array[digest] = 1 #then sets that bit to 1
    def check(self, item):
        return all(self.bit_array[func(item) % self.size] for func in self.hash_functions) #checks if item is in filter
    
# hashes do not have to change


In [ ]:
# hashes do not have to change
# create new blooms
# are these different ways to hash? (sha256, blake2b, sha3_256)

#def my_hash(s):
    return int(sha256(s.lower().encode()).hexdigest(), 16) % size
def my_hash2(s):
    return int(blake2b(s.lower().encode()).hexdigest(), 16) % size
def my_hash3(s):
    return int(sha3_256(s.lower().encode()).hexdigest(), 16) % size

for word in words: 
    index1 = my_hash(word)
    index2 = my_hash2(word)
    index3 = my_hash3(word)
    bits[index1] = 1
    bits[index2] = 1
    bits[index3] = 1 #hashes all words in words and set 1 to dictate there is something there
    
#Asked ChatGPT is I answered all parts of the question and modified code (define size) 

bloom1 = BloomFilter_new(1000000,my_hash)
bloom2 = BloomFilter_new(1000000, (my_hash, my_hash2))
bloom3 = BloomFilter_new(1000000,(my_hash, my_hash2, my_hash3))

print(bloom1)
print(bloom2)
print(bloom3)


In [ ]:
#test all 3 with flower
# good_bloom('floeer', 'flower', bloom1)

['floter']


'This is a bad outcome'

In [ ]:
#good_bloom('floeer', 'flower', bloom2)


['floter']


'This is a bad outcome'

In [ ]:
#good_bloom('floeer', 'flower', bloom3)

['floter']


'This is a bad outcome'

In [ ]:
#track output/performance of each bloom 
#asked ChatGPT how to modify to keep track of good outcome and bad outcome

# def track_bloom_one(typo_pairs, bloom):
    correct_one_hash = []
    wrong_one_hash = []
    for typed_word, correct_word in typo_pairs:
        result = good_bloom(typed_word, correct_word, bloom)
        if "good" in result.lower():
            correct_one_hash.append('good')
        else:
            wrong_one_hash.append('bad')
    return correct_one_hash, wrong_one_hash

In [ ]:
#typo_pairs = [('floeer', 'flower')]

#track_bloom_one(typo_pairs, bloom1)

['floter']


([], ['bad'])

In [ ]:
# Track the rate of false positives (incorrect words misidentified as correct) and good suggestions (as defined above). 
# Plot the results.
# Approximate how many bits are necessary to acheive 85% good suggestions with each combination of 1, 2, 3, hashes

In [ ]:
#new new

# class BloomFilter_new:
    def __init__(self, size, hash_functions): #now uses any hash function
        #items_count number of items expected to be sotred in bloom filter
        self.size = size #set size asspecified in bloom
        self.bit_array = bitarray(self.size) #bit array of given size
        self.bit_array.setall(0) #initialize all bits as 0
        self.hash_functions = hash_functions #allows provided hashing

    def add(self, item):
        for func in self.hash_functions:
            digest = func(item) % self.size #each hash produces integer from item within bounds of bit array
            self.bit_array[digest] = True #then sets that bit to True to check later for graph
    def check(self, item):
        #check for item in filteer
        for active_hash in self.hash_functions:
            if active_hash #hash the word then go through bit array and see if hash is there 
                #if any of bit is False, not present; else possibility it exists
                return True
        return False


In [ ]:
#the word is correct 
# the word is misidentified but spell checker outputed as correct
# the word is mididentified and spell checker told as misidentified 

#if spell checker works and tells if misidentified 

3